In [3]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

df = pd.read_csv('../data/AmesHousing.csv')
print(f'Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns')

Loaded dataset: 2930 rows, 82 columns


In [4]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
})

missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing_df)
print(f'\nTotal columns with missing values: {len(missing_df)}')

                Missing Count  Missing %
Pool QC                  2917      99.56
Misc Feature             2824      96.38
Alley                    2732      93.24
Fence                    2358      80.48
Mas Vnr Type             1775      60.58
Fireplace Qu             1422      48.53
Lot Frontage              490      16.72
Garage Cond               159       5.43
Garage Qual               159       5.43
Garage Finish             159       5.43
Garage Yr Blt             159       5.43
Garage Type               157       5.36
Bsmt Exposure              83       2.83
BsmtFin Type 2             81       2.76
Bsmt Cond                  80       2.73
Bsmt Qual                  80       2.73
BsmtFin Type 1             80       2.73
Mas Vnr Area               23       0.78
Bsmt Half Bath              2       0.07
Bsmt Full Bath              2       0.07
BsmtFin SF 1                1       0.03
Garage Cars                 1       0.03
Garage Area                 1       0.03
Total Bsmt SF   

In [5]:
# Drop columns with >50% missing values
threshold = 50
cols_to_drop = missing_df[missing_df['Missing %'] > threshold].index.tolist()

print(f'Columns to drop (>50% missing): {cols_to_drop}')
df = df.drop(columns=cols_to_drop)  # use assignment instead of inplace
print(f'Dropped {len(cols_to_drop)} columns')
print(f'New shape: {df.shape}')

Columns to drop (>50% missing): ['Pool QC', 'Misc Feature', 'Alley', 'Fence', 'Mas Vnr Type']
Dropped 5 columns
New shape: (2930, 77)


In [6]:
# Fill missing numeric values with median
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)  # fixed: no chained inplace
        print(f'Filled {col} with median: {median_val}')

print('\nAll numeric missing values filled')

Filled Lot Frontage with median: 68.0
Filled Mas Vnr Area with median: 0.0
Filled BsmtFin SF 1 with median: 370.0
Filled BsmtFin SF 2 with median: 0.0
Filled Bsmt Unf SF with median: 466.0
Filled Total Bsmt SF with median: 990.0
Filled Bsmt Full Bath with median: 0.0
Filled Bsmt Half Bath with median: 0.0
Filled Garage Yr Blt with median: 1979.0
Filled Garage Cars with median: 2.0
Filled Garage Area with median: 480.0

All numeric missing values filled


In [7]:
# Fill missing categorical values with mode
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)  # fixed: no chained inplace
        print(f'Filled {col} with mode: {mode_val}')

print('\nAll categorical missing values filled')

Filled Bsmt Qual with mode: TA
Filled Bsmt Cond with mode: TA
Filled Bsmt Exposure with mode: No
Filled BsmtFin Type 1 with mode: GLQ
Filled BsmtFin Type 2 with mode: Unf
Filled Electrical with mode: SBrkr
Filled Fireplace Qu with mode: Gd
Filled Garage Type with mode: Attchd
Filled Garage Finish with mode: Unf
Filled Garage Qual with mode: TA
Filled Garage Cond with mode: TA

All categorical missing values filled


In [8]:
# Verify no missing values left
total_missing = df.isnull().sum().sum()
print(f'Total missing values remaining: {total_missing}')

if total_missing == 0:
    print('No missing values - all clean!')
else:
    print('Still some missing values')

Total missing values remaining: 0
No missing values - all clean!


In [9]:
# Remove extreme outliers from SalePrice using IQR method
Q1 = df['SalePrice'].quantile(0.25)
Q3 = df['SalePrice'].quantile(0.75)
IQR = Q3 - Q1

# Fixed: use symmetric 1.5x bounds (was asymmetric 1.2x/1.5x before)
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

before = df.shape[0]
df = df[(df['SalePrice'] >= lower) & (df['SalePrice'] <= upper)]
after = df.shape[0]

print(f'Rows before: {before}')
print(f'Rows after: {after}')
print(f'Outliers removed: {before - after}')

Rows before: 2930
Rows after: 2793
Outliers removed: 137


In [10]:
# Remove duplicate rows
before = df.shape[0]
df = df.drop_duplicates()  # fixed: use assignment instead of inplace
after = df.shape[0]

print(f'Duplicates removed: {before - after}')
print(f'Final shape: {df.shape}')


Duplicates removed: 0
Final shape: (2793, 77)


In [12]:
# save clean data
import os
os.makedirs('..data',exist_ok=True)
df.to_csv('../data/clean_data.csv',index=False)
print('saved')
print(f'Final dataset:{df.shape[0]}rows,{df.shape[1]}columns')

saved
Final dataset:2793rows,77columns
